# Phase 1-2: Data Understanding & Cleaning

Load the raw retail export, audit data quality (missing, duplicates, outliers, invalid values) and apply a documented cleaning pipeline with feature engineering — every decision is justified by its business impact.

## 1. Load raw data
We load the raw export (which intentionally contains realistic data-quality
issues: missing values, duplicates, impossible dates, price typos, inconsistent
casing, and non-completed orders).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
df = pd.read_csv('../data/raw/raw_retail_sales.csv')
df.head(3)

,Order_ID,Order_Date,Customer_ID,Product_ID,Product_Name,Category,Sub_Category,Quantity,Unit_Price,Discount,...,Profit_Margin,Customer_Name,Customer_Segment,Age,Gender,City,State,Region,Payment_Method,Order_Status
0,10000,2021-12-05,1,24,StapleCo Storage & Organization 2,Office Supplies,Storage & Organization,3,89.34,0.1734,...,0.2137,Andrew Lewis,Corporate,42.0,Female,Phoenix,Arizona,West,Net Banking,Cancelled
1,10000,2021-12-05,1,48,CoreX Peripheral 6,Technology,Peripherals,2,151.29,0.2509,...,0.0521,Andrew Lewis,Corporate,42.0,Female,Phoenix,Arizona,West,Net Banking,Cancelled
2,10000,2021-12-05,1,70,Oakline Office Desk 4,Furniture,Office Desks,4,846.29,0.0,...,0.3600,Andrew Lewis,Corporate,42.0,Female,Phoenix,Arizona,West,Net Banking,Cancelled


## 2. Data understanding (Phase 1)
### Dimensions & data types

In [2]:
print('Rows:', df.shape[0], '| Columns:', df.shape[1])
df.dtypes

Rows: 117049 | Columns: 23


Order_ID              int64
Order_Date           object
Customer_ID           int64
Product_ID            int64
Product_Name         object
Category             object
Sub_Category         object
Quantity              int64
Unit_Price          float64
Discount             object
Sales               float64
Cost                float64
Profit              float64
Profit_Margin       float64
Customer_Name        object
Customer_Segment     object
Age                 float64
Gender               object
City                 object
State                object
Region               object
Payment_Method       object
Order_Status         object
dtype: object

In [3]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Order_ID,117049.0,NaN,NaN,NaN,27841.678562,10409.115747,10000.0,18732.0,27699.0,36897.0,45754.0
Order_Date,117049,1464,2024-12-21,202,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer_ID,117049.0,NaN,NaN,NaN,896.920691,536.286459,1.0,411.0,901.0,1367.0,1800.0
Product_ID,117049.0,NaN,NaN,NaN,47.009705,31.594872,1.0,17.0,44.0,78.0,101.0
Product_Name,117049,101,StapleCo Paper 5,1998,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Category,117049,8,Office Supplies,42508,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Sub_Category,117049,20,Paper,11612,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,117049.0,NaN,NaN,NaN,2.996583,1.426534,-5.0,2.0,3.0,4.0,5.0
Unit_Price,117049.0,NaN,NaN,NaN,241.507534,380.332713,-1873.59,40.19,122.16,252.03,22318.9
Discount,117049,4497,0.0,35330,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Missing values

In [4]:
missing = df.isna().sum()
missing[missing > 0]

Age     1837
City    1117
dtype: int64

### Duplicates & invalid values

In [5]:
print('Duplicate rows:', df.duplicated().sum())
print('Negative Quantity:', (df['Quantity'] < 0).sum())
print('Negative Unit_Price:', (df['Unit_Price'] < 0).sum())
print('Impossible dates (month 00/13):',
      df['Order_Date'].astype(str).str[5:7].isin(['00', '13']).sum())
print('\nOrder status counts:'); print(df['Order_Status'].value_counts())

Duplicate rows: 367
Negative Quantity: 114
Negative Unit_Price: 59
Impossible dates (month 00/13): 140

Order status counts:
Order_Status
Completed    108698
Cancelled      3614
Returned       2916
Pending        1821
Name: count, dtype: int64


### Outlier check (descriptive stats on numeric columns)
`Profit_Margin` at -0.5 already hints that heavy discounts can make lines
unprofitable — we will verify this formally in the statistics notebook.

In [6]:
df[['Quantity', 'Unit_Price', 'Discount', 'Sales', 'Profit', 'Profit_Margin']].describe().round(2)

,Quantity,Unit_Price,Sales,Profit,Profit_Margin
count,117049.00,117049.00,117049.00,117049.00,117049.00
mean,3.00,241.51,638.64,151.72,0.29
std,1.43,380.33,1079.00,256.49,0.14
min,-5.00,-1873.59,2.10,-2949.44,-0.49
25%,2.00,40.19,94.26,22.16,0.22
50%,3.00,122.16,250.35,64.80,0.32
75%,4.00,252.03,682.52,186.90,0.39
max,5.00,22318.90,11159.45,2033.75,0.49


## 3. Cleaning pipeline (Phase 2)
Each transformation below is paired with its business justification.

### 3.1 Dates — parse & drop impossible dates
`errors='coerce'` turns unparseable rows into `NaT`, which we drop
(~0.2% of data). These rows cannot be repaired reliably.

In [7]:
df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce')
df = df.dropna(subset=['Order_Date'])
df['Order_Date'].describe()

count                           116840
mean     2023-02-25 08:42:41.040739584
min                2021-01-01 00:00:00
25%                2022-02-20 00:00:00
50%                2023-03-08 00:00:00
75%                2024-02-18 00:00:00
max                2024-12-31 00:00:00
Name: Order_Date, dtype: object

### 3.2 Drop negative quantities (invalid transaction lines)

In [8]:
df = df[df['Quantity'] >= 0]

### 3.3 Fix Unit_Price typos using the price identity
Retail price identity: `Sales = Quantity * Unit_Price * (1 - Discount)`.
If the recorded price disagrees with the identity (e.g. a 10x typo), we
recalculate it. Negative prices that cannot be derived are dropped.

In [9]:
df['Discount'] = pd.to_numeric(df['Discount'], errors='coerce')
has_disc = df['Discount'].notna()
expected_price = np.where(has_disc,
    df['Sales'] / np.maximum(df['Quantity'] * (1 - df['Discount']), 1e-9), np.nan)
inconsistent = has_disc & (np.abs(df['Unit_Price'] - expected_price) > 0.01 * expected_price)
print('Unit prices recalculated:', int(inconsistent.sum()))
df.loc[inconsistent, 'Unit_Price'] = np.round(expected_price[inconsistent], 2)
df = df[df['Unit_Price'] > 0]

Unit prices recalculated: 208


### 3.4 Derive missing discounts from the price identity
Instead of guessing missing discounts we derive them:
`Discount = 1 - Sales / (Qty * Unit_Price)`, clamped to a sensible [0, 0.9].

In [10]:
missing = df['Discount'].isna()
derived = 1 - df['Sales'] / (df['Quantity'] * df['Unit_Price'])
df.loc[missing, 'Discount'] = np.clip(derived[missing], 0, 0.9)
print('Discounts derived:', int(missing.sum()))
print('Discount stats after:', df['Discount'].describe().round(4).to_dict())

Discounts derived: 1443
Discount stats after: {'count': 116724.0, 'mean': 0.1113, 'std': 0.1164, 'min': 0.0, '25%': 0.0, '50%': 0.0834, '75%': 0.1839, 'max': 0.45}


### 3.5 Remove exact duplicates (would double-count revenue)

In [11]:
print('Duplicates removed:', int(df.duplicated().sum()))
df = df.drop_duplicates()

Duplicates removed: 370


### 3.6 Standardise categorical values
Lowercase `'technology'` entries would create phantom categories in
`GROUP BY`; we normalise to Title Case.

In [12]:
df['Category'] = df['Category'].str.strip().str.title()
df['Sub_Category'] = df['Sub_Category'].str.strip().str.title()
df['Payment_Method'] = df['Payment_Method'].str.strip()
df['Order_Status'] = df['Order_Status'].str.strip().str.title()
df['Category'].value_counts()

Category
Office Supplies     42509
Home & Lifestyle    30568
Technology          24613
Furniture           18664
Name: count, dtype: int64

### 3.7 Impute missing Age & City
Age differs by segment (Corporate buyers skew older), so we impute with the
**segment median**. City always belongs to a known state, so we impute with
the **state mode** (most common city in that state).

In [13]:
df['Age'] = df.groupby('Customer_Segment')['Age'].transform(lambda s: s.fillna(s.median()))
df['City'] = df.groupby('State')['City'].transform(
    lambda s: s.fillna(s.mode().iloc[0] if len(s.mode()) else 'Unknown'))
print('Missing values remaining:', int(df.isna().sum().sum()))

Missing values remaining: 0


### 3.8 Business rule — Completed orders only
Cancelled / Returned / Pending orders generate **no revenue**. We keep the
full history in the SQL database but exclude them from revenue analytics.

In [14]:
print('Excluded (non-Completed):', int((df['Order_Status'] != 'Completed').sum()))
df = df[df['Order_Status'] == 'Completed']

Excluded (non-Completed): 8309


### 3.9 Feature engineering
Derived variables unlock time-based and categorical analysis.

In [15]:
df['Order_Year'] = df['Order_Date'].dt.year
df['Order_Month'] = df['Order_Date'].dt.month
df['Order_Day'] = df['Order_Date'].dt.day
df['Day_Of_Week'] = df['Order_Date'].dt.day_name()

age_bins = [0, 25, 35, 45, 55, 65, 100]
df['Age_Group'] = pd.cut(df['Age'], bins=age_bins,
                         labels=['18-24', '25-34', '35-44', '45-54', '55-64', '65+'], right=False)
df['Discount_Band'] = pd.cut(df['Discount'], bins=[-0.001, 0.001, 0.10, 0.20, 0.30, 1.0],
                             labels=['No Discount', '0-10%', '10-20%', '20-30%', '30%+'],
                             include_lowest=True)
df['Revenue'] = df['Sales']

## 4. Reconciliation check
Every surviving line must satisfy `Sales = Qty * Unit_Price * (1 - Discount)`
to within 1 cent (rounding). This is our QA gate before analysis.

In [16]:
resid = (df['Sales'] - df['Quantity'] * df['Unit_Price'] * (1 - df['Discount'])).abs()
print('Lines failing reconciliation (>$0.05):', int((resid > 0.05).sum()))
print('Max residual ($):', round(float(resid.max()), 3))

Lines failing reconciliation (>$0.05): 0
Max residual ($): 0.005


## 5. Export cleaned data

In [17]:
df.to_csv('../data/cleaned/retail_sales_clean.csv', index=False)
print('Saved', len(df), 'rows x', df.shape[1], 'columns')

Saved 108045 rows x 30 columns
